# Poker Hand Examples

`classify_hand` labels a 5-card hand with its best poker category, and a family of `is_...` predicates check for a specific category. Each one takes a hand drawn from `DeckOfCards(size=5)`, so they drop straight into the usual `RV(deck).apply(...)` flow for simulating probabilities.

In [1]:
from symbulate import *

---
## A poker hand

`DeckOfCards(size=5)` deals five cards without replacement. Each card is a `(rank, suit)` tuple.

In [2]:
deck = DeckOfCards(size=5)
deck.draw()

(('A', 'Diamonds'), (2, 'Hearts'), ('J', 'Clubs'), (2, 'Spades'), ('J', 'Hearts'))

---
## classify_hand

`classify_hand(hand)` returns the hand's single best category, one of the ten names in `POKER_HANDS`.

In [3]:
POKER_HANDS

('royal flush',
 'straight flush',
 'four of a kind',
 'full house',
 'flush',
 'straight',
 'three of a kind',
 'two pair',
 'pair',
 'high card')

In [4]:
royal = [(10, 'Hearts'), ('J', 'Hearts'), ('Q', 'Hearts'),
         ('K', 'Hearts'), ('A', 'Hearts')]
classify_hand(royal)

'royal flush'

Apply it to many simulated hands and tabulate to recover the classic poker-hand distribution.

In [5]:
RV(deck).apply(classify_hand).sim(50000).tabulate(normalize=True)

flush,0.00166
four of a kind,0.00026
full house,0.00132
high card,0.50078
pair,0.42156
straight,0.00376
three of a kind,0.02252
two pair,0.04814
Total,1.0


---
## The `is_...` predicates

One predicate per category: `is_royal_flush`, `is_straight_flush`, `is_four_of_a_kind`, `is_full_house`, `is_flush`, `is_straight`, `is_three_of_a_kind`, `is_two_pair`, `is_pair`, `is_high_card`. Each returns `True`/`False`, so averaging over many hands estimates that hand's probability.

In [6]:
RV(deck).apply(is_full_house).sim(50000).mean()   # exact value ≈ 0.00144

0.0013

In [7]:
RV(deck).apply(is_pair).sim(10000).mean()         # exact value ≈ 0.4226

0.4241

---
## Categories are mutually exclusive

`classify_hand` reports the *best* category, so a full house is not also counted as a pair. The estimated probabilities therefore partition cleanly and sum to 1.

In [8]:
full_house = [(3, 'Clubs'), (3, 'Hearts'), (3, 'Spades'),
              (8, 'Clubs'), (8, 'Diamonds')]
classify_hand(full_house), is_full_house(full_house), is_pair(full_house)

('full house', True, False)

---
## Conditional probability

Build events with `.apply()` and condition one on another with `|`. Here: the chance of a flush *given* that every card is red. Both events come from the same `hand`, so they refer to the same dealt cards.

In [9]:
hand = RV(deck)
flush = hand.apply(is_flush)
red = hand.apply(lambda h: all(card[1] in ('Hearts', 'Diamonds') for card in h))
(flush | (red == True)).sim(5000).mean()          # P(flush | all red) ≈ 0.040

0.0376

Compare to the unconditional chance of a flush, about `0.00198` — restricting to red cards (two suits instead of four) makes a flush roughly 20 times more likely.

In [10]:
flush.sim(50000).mean()

0.00178